# SafeStreet AI - XGBoost V2

## Objective

Train the improved XGBoost model using the Feature Engineering V2 dataset.

---

### Input

- processed_crime_v2.parquet

### Output

- safestreet_xgb_v2.joblib
- safestreet_xgb_v2_metadata.json

### Experiment

Experiment 06

XGBoost + Feature Engineering V2

In [1]:
# ==========================================
# Imports
# ==========================================

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

import joblib

import json

print("All libraries imported successfully!")

All libraries imported successfully!


# ==========================================
# Experiment Configuration
# ==========================================

In [2]:
# ==========================================
# Experiment Configuration
# ==========================================

# Dataset
USE_SAMPLE = True
SAMPLE_SIZE = 1_000_000

# Train/Test Split
TEST_SIZE = 0.20
RANDOM_STATE = 42

# XGBoost Parameters
N_ESTIMATORS = 300
MAX_DEPTH = 8
LEARNING_RATE = 0.10
SUBSAMPLE = 0.80
COLSAMPLE_BYTREE = 0.80

# Output
MODEL_NAME = "safestreet_xgb_v2"

print("=" * 60)
print("Experiment Configuration Loaded Successfully")
print("=" * 60)

print(f"Sample Size      : {SAMPLE_SIZE:,}")
print(f"Test Size        : {TEST_SIZE}")
print(f"Random State     : {RANDOM_STATE}")
print(f"Model Name       : {MODEL_NAME}")

Experiment Configuration Loaded Successfully
Sample Size      : 1,000,000
Test Size        : 0.2
Random State     : 42
Model Name       : safestreet_xgb_v2


Configuration completed"

In [4]:
# ==========================================
# Load Feature Engineering V2 Dataset
# ==========================================

from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/processed_crime_v2.parquet")

df = pd.read_parquet(DATA_PATH)

print("=" * 60)
print("Feature Engineering V2 Dataset Loaded Successfully")
print("=" * 60)

print(f"Dataset Shape: {df.shape}")

df.head()

Feature Engineering V2 Dataset Loaded Successfully
Dataset Shape: (7758698, 17)


,Latitude,Longitude,Hour,DayOfWeek,Month,IsWeekend,District,Community Area,Location Description,RiskLevel,TimeOfDay,IsNight,IsBusinessHours,Hour_sin,Hour_cos,Month_sin,Month_cos
0,41.813999,-87.598138,18,3,9,0,2.0,39.0,RESIDENCE,Low,Evening,0,0,-1.000000,-1.836970e-16,-1.0,-1.836970e-16
1,41.954584,-87.648376,22,5,9,1,19.0,3.0,STREET,Low,Night,1,0,-0.500000,8.660254e-01,-1.0,-1.836970e-16
2,41.808521,-87.626066,1,6,9,1,2.0,38.0,VACANT LOT/LAND,High,Night,1,0,0.258819,9.659258e-01,-1.0,-1.836970e-16
3,41.885759,-87.713588,2,6,9,1,11.0,27.0,SIDEWALK,High,Night,1,0,0.500000,8.660254e-01,-1.0,-1.836970e-16
4,41.947100,-87.662116,2,6,9,1,19.0,6.0,SIDEWALK,Low,Night,1,0,0.500000,8.660254e-01,-1.0,-1.836970e-16


In [5]:
# ==========================================
# Verify Feature Engineering V2 Columns
# ==========================================

expected_features = [
    "TimeOfDay",
    "IsNight",
    "IsBusinessHours",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos",
]

print("Checking Feature Engineering V2 columns...\n")

missing_features = []

for feature in expected_features:
    if feature in df.columns:
        print(f"✓ {feature}")
    else:
        print(f"✗ {feature}")
        missing_features.append(feature)

if not missing_features:
    print("\nAll Feature Engineering V2 columns are available.")
else:
    print("\nMissing Features:")
    print(missing_features)

Checking Feature Engineering V2 columns...

✓ TimeOfDay
✓ IsNight
✓ IsBusinessHours
✓ Hour_sin
✓ Hour_cos
✓ Month_sin
✓ Month_cos

All Feature Engineering V2 columns are available.


# ==========================================
# Create Development Sample
# ==========================================

In [6]:
# ==========================================
# Create Development Sample
# ==========================================

if USE_SAMPLE:

    train_df, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        stratify=df["RiskLevel"],
        random_state=RANDOM_STATE,
    )

else:
    train_df = df.copy()

print("=" * 60)
print("Development Dataset Created")
print("=" * 60)

print(f"Training Dataset Shape : {train_df.shape}")

print("\nRisk Level Distribution (%)")

print(
    train_df["RiskLevel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Development Dataset Created
Training Dataset Shape : (1000000, 17)

Risk Level Distribution (%)
RiskLevel
Low       58.53
High      29.61
Medium    11.87
Name: proportion, dtype: float64


# ==========================================
# Feature Selection
# ==========================================

In [7]:
# ==========================================
# Feature Selection
# ==========================================

FEATURES = [

    # Original Features
    "Latitude",
    "Longitude",
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "District",
    "Community Area",
    "Location Description",

    # Feature Engineering V2
    "TimeOfDay",
    "IsNight",
    "IsBusinessHours",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos",
]

TARGET = "RiskLevel"

X = train_df[FEATURES]
y = train_df[TARGET]

print("=" * 60)
print("Feature Selection Completed")
print("=" * 60)

print(f"Number of Features : {len(FEATURES)}")
print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Shape : {y.shape}")

print("\nSelected Features:\n")

for i, feature in enumerate(FEATURES, start=1):
    print(f"{i:2d}. {feature}")

Feature Selection Completed
Number of Features : 16
Feature Matrix Shape : (1000000, 16)
Target Shape : (1000000,)

Selected Features:

 1. Latitude
 2. Longitude
 3. Hour
 4. DayOfWeek
 5. Month
 6. IsWeekend
 7. District
 8. Community Area
 9. Location Description
10. TimeOfDay
11. IsNight
12. IsBusinessHours
13. Hour_sin
14. Hour_cos
15. Month_sin
16. Month_cos


# ==========================================
# Identify Feature Types
# ==========================================

In [9]:
# ==========================================
# Identify Feature Types
# ==========================================

# Categorical Features
categorical_features = (
    X.select_dtypes(include=["object", "category", "string"])
    .columns
    .tolist()
)

# Numerical Features
numerical_features = (
    X.select_dtypes(include=["number"])
    .columns
    .tolist()
)

print("=" * 60)
print("Feature Type Identification")
print("=" * 60)

print(f"Categorical Features ({len(categorical_features)}):")
for feature in categorical_features:
    print(f"  • {feature}")

print()

print(f"Numerical Features ({len(numerical_features)}):")
for feature in numerical_features:
    print(f"  • {feature}")

Feature Type Identification
Categorical Features (2):
  • Location Description
  • TimeOfDay

Numerical Features (14):
  • Latitude
  • Longitude
  • Hour
  • DayOfWeek
  • Month
  • IsWeekend
  • District
  • Community Area
  • IsNight
  • IsBusinessHours
  • Hour_sin
  • Hour_cos
  • Month_sin
  • Month_cos


# ==========================================
# Build Preprocessing Pipeline
# ==========================================

In [10]:
# ==========================================
# Build Preprocessing Pipeline
# ==========================================

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Combined preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("=" * 60)
print("Preprocessing Pipeline Created Successfully")
print("=" * 60)

print(preprocessor)

Preprocessing Pipeline Created Successfully
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['Latitude', 'Longitude', 'Hour', 'DayOfWeek',
                                  'Month', 'IsWeekend', 'District',
                                  'Community Area', 'IsNight',
                                  'IsBusinessHours', 'Hour_sin', 'Hour_cos',
                                  'Month_sin', 'Month_cos']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ign

# ==========================================
# Build XGBoost Pipeline
# ==========================================

In [11]:
# ==========================================
# Build XGBoost Pipeline
# ==========================================

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            XGBClassifier(
                objective="multi:softmax",
                num_class=3,

                n_estimators=N_ESTIMATORS,
                max_depth=MAX_DEPTH,
                learning_rate=LEARNING_RATE,

                subsample=SUBSAMPLE,
                colsample_bytree=COLSAMPLE_BYTREE,

                random_state=RANDOM_STATE,

                eval_metric="mlogloss",

                n_jobs=-1,
            ),
        ),
    ]
)

print("=" * 60)
print("XGBoost Pipeline Created Successfully")
print("=" * 60)

print(xgb_model)

XGBoost Pipeline Created Successfully
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Latitude', 'Longitude',
                                                   'Hour', 'DayOfWeek', 'Month',
                                                   'IsWeekend', 'District',
                                                   'Community Area', 'IsNight',
                                                   'IsBusinessHours',
                                                   'Hour_sin', 'Hour_cos',
                                                   'Month_sin', 'Month_cos']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                  

# ==========================================
# Train/Test Split
# ==========================================

In [12]:
# ==========================================
# Train/Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("=" * 60)
print("Train/Test Split Completed")
print("=" * 60)

print(f"Training Samples : {len(X_train):,}")
print(f"Testing Samples  : {len(X_test):,}")

print("\nTraining Distribution")

print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting Distribution")

print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Train/Test Split Completed
Training Samples : 800,000
Testing Samples  : 200,000

Training Distribution
RiskLevel
Low       58.53
High      29.61
Medium    11.87
Name: proportion, dtype: float64

Testing Distribution
RiskLevel
Low       58.53
High      29.61
Medium    11.87
Name: proportion, dtype: float64


Train/Test Split completed

In [13]:
# ==========================================
# Encode Target Labels
# ==========================================

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("=" * 60)
print("Target Labels Encoded Successfully")
print("=" * 60)

print("Class Mapping:\n")

for i, label in enumerate(label_encoder.classes_):
    print(f"{i} -> {label}")

Target Labels Encoded Successfully
Class Mapping:

0 -> High
1 -> Low
2 -> Medium


# ==========================================
# Train XGBoost V2
# ==========================================

In [14]:
# ==========================================
# Train XGBoost V2
# ==========================================

import time

print("=" * 60)
print("Training XGBoost V2...")
print("=" * 60)

start_time = time.time()

xgb_model.fit(
    X_train,
    y_train_encoded
)

end_time = time.time()

training_time = end_time - start_time

print("\nTraining Completed Successfully!")

print(f"Training Time : {training_time:.2f} seconds")
print(f"Training Time : {training_time/60:.2f} minutes")

Training XGBoost V2...

Training Completed Successfully!
Training Time : 21.55 seconds
Training Time : 0.36 minutes


# ==========================================
# Evaluate XGBoost V2
# ==========================================

In [15]:
# ==========================================
# Evaluate XGBoost V2
# ==========================================

print("=" * 60)
print("Evaluating XGBoost V2...")
print("=" * 60)

# Predict encoded labels
y_pred_encoded = xgb_model.predict(X_test)

# Convert back to original labels
y_pred = label_encoder.inverse_transform(y_pred_encoded)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy : {accuracy:.4f}")
print(f"Accuracy : {accuracy * 100:.2f}%")

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

Evaluating XGBoost V2...

Accuracy : 0.6226
Accuracy : 62.26%

Classification Report

              precision    recall  f1-score   support

        High       0.57      0.32      0.41     59211
         Low       0.64      0.88      0.74    117053
      Medium       0.56      0.08      0.14     23736

    accuracy                           0.62    200000
   macro avg       0.59      0.43      0.43    200000
weighted avg       0.61      0.62      0.57    200000


Confusion Matrix

[[ 19171  39826    214]
 [ 12334 103375   1344]
 [  2394  19376   1966]]


# ==========================================
# Save XGBoost V2 Model
# ==========================================

In [16]:
# ==========================================
# Save XGBoost V2 Model
# ==========================================

MODEL_DIR = Path("../saved_models")
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "safestreet_xgb_v2.joblib"

joblib.dump(xgb_model, model_path)

print("Model saved successfully!")
print(model_path.resolve())

Model saved successfully!
D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\saved_models\safestreet_xgb_v2.joblib


In [17]:
# ==========================================
# Save Label Encoder
# ==========================================

encoder_path = MODEL_DIR / "label_encoder_v2.joblib"

joblib.dump(label_encoder, encoder_path)

print("Label Encoder saved successfully!")
print(encoder_path.resolve())

Label Encoder saved successfully!
D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\saved_models\label_encoder_v2.joblib


In [18]:
# ==========================================
# Save Metadata
# ==========================================

metadata = {
    "model_name": MODEL_NAME,
    "dataset": "processed_crime_v2.parquet",
    "experiment": "E06",
    "algorithm": "XGBoost",
    "features": FEATURES,
    "accuracy": round(accuracy, 4),
    "training_time_seconds": round(training_time, 2),
    "random_state": RANDOM_STATE,
}

metadata_path = MODEL_DIR / "safestreet_xgb_v2_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved successfully!")
print(metadata_path.resolve())

Metadata saved successfully!
D:\SOUMIT MANNA\IEM\Internship\Summer Internship\SafeStreet\ml_backend\saved_models\safestreet_xgb_v2_metadata.json
